# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook walks through loading, exploring, and processing the [FAIR^2 protein abundance and modification dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s (identifiers).

In [ ]:
# List all record sets and their fields by @id
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the Croissant schema metadata. Most likely, records are described as 'ExperimentDataset'. Let's fetch all datasets and check their ids and fields.")
    # Fallback: in this schema, ExperimentDataset is used.
    exp_record_sets = [rs for rs in dataset.iter_entities() if '@type' in rs and rs['@type'] == 'Dataset']
    print(f"Found {len(exp_record_sets)} datasets.")
    for rs in exp_record_sets:
        print(f"@id: {rs.get('@id')}")
        print(f"  Name: {rs.get('name','')}")
        print(f"  Description: {rs.get('description','')}")
        fields = rs.get('field') or rs.get('fields') or []
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields (by @id):")
        for f in fields:
            print(f"    - {f.get('@id') if isinstance(f, dict) else f}")
        print()
else:
    for rs in record_sets:
        print(f"@id: {rs.id}")
        print(f"  Name: {rs.name if hasattr(rs, 'name') else ''}")
        print(f"  Fields (by @id):")
        for field in rs.fields:
            print(f"    - {field.id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this dataset, the primary record set appears to be `'dv:ExperimentDataset'` (see the Croissant), typically referencing the main tabular data file.

In [ ]:
# Set up the main record set for this dataset
main_record_set_id = 'dv:ExperimentDataset'
record_sets = [main_record_set_id]
dataframes = {}

# Try to extract records for each record set
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))  # each record is a dict
        if not records:
            print(f"Warning: No records loaded for {record_set}.")
        dataframes[record_set] = pd.DataFrame(records)
    except Exception as e:
        print(f"Could not load records for {record_set}: {e}")

if main_record_set_id in dataframes:
    print("DataFrame columns for record set '@id':", main_record_set_id)
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations such as removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

**Note:** All fields and columns are referenced by their `@id`.

In [ ]:
# For demonstration, analyze a numeric field such as 'cr:coverage_percent' (if present)
import numpy as np

df = dataframes[main_record_set_id]
numeric_field_id = None
for col in df.columns:
    if 'coverage' in col and 'percent' in col:
        numeric_field_id = col
        break
if numeric_field_id is None and len(df.select_dtypes(include=np.number).columns)>0:
    numeric_field_id = df.select_dtypes(include=np.number).columns[0]

if numeric_field_id is not None:
    print(f"Analyzing numeric field (by @id): {numeric_field_id}")
    # Ensure numeric conversion in case it's read as string
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75)  # use 75th percentile as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Sample of normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by protein name or accession if present
    group_field_id = None
    for candidate in ['cr:accession', 'cr:description', 'cr:protein_name']:
        if candidate in df.columns:
            group_field_id = candidate
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id} (showing top 5):")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If protein/description available, plot top N by abundance
    if group_field_id and group_field_id in df.columns:
        topn = 10
        means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(topn)
        plt.figure(figsize=(10,6))
        sns.barplot(y=means.index, x=means.values, orient='h')
        plt.title(f"Top {topn} proteins by mean {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the [FAIR^2 protein dataset](https://doi.org/10.71728/senscience.vq4a-28xa) using `mlcroissant`. We extracted records using Croissant `@id`s, identified main data fields, and performed sample EDA including filtering by a coverage field, normalization, and visualization. Use the field and record set `@id`s for reproducible references and easy integration in future analyses.